# VL13 - Parameter-Efficient Fine-Tunning
# Part 1 - Training

### Step 0 — Setup

We install the libraries needed for supervised fine-tuning (SFT) with LoRA:
- `transformers`: model + tokenizer
- `datasets`: train/test split
- `peft`: LoRA adapters (train only a few parameters)
- `trl`: a simple SFT trainer loop

````bash
$ pip install peft trl bitsandbytes accelerate
````

Then we import everything we need.

In [ ]:
import os
import re
import json
import random
import numpy as np
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

## 1. Introducing the task
We will fine tune a LLM to perform structured information extraction from a receipt. We assume the receipts are already OCRed, and we work directly with text. For this purpose, we will:
- download and normalize the dataset
- prepare the data for training, as intruction -> output examples
- fine-tune the model using LoRA
- save the model

### 1.1 Loading the dataset

We load again the SROIE receipt dataset from a JSON file.
Each row corresponds to **one receipt**, with:

- `transcription` → OCR text (model input)
- `company`, `date`, `address`, `total` → gold fields (ground truth)

Fine-tuning needs supervised examples:
- input: receipt text
- output: the correct JSON

In [ ]:
import json

PATH = "../../data/sroie/icdar-2019-sroie.json"

with open(PATH, "r", encoding="utf-8") as f:
    sroie = json.load(f)

FIELDS = ["company", "date", "address", "total"] # fields to be extracted

items = sroie["items"]
items[0].keys()


### 1.2 Preparing the data
We normalise the data, making sure all expected fields are there. If not, we will them with "NOT ANSWERABLE".

In [ ]:
import json

def normalize_fields(fields: dict):
    """
    Ensure:
    - all required keys exist
    - missing values are marked as NOT ANSWERABLE
    - output is a JSON string (stable format)
    """
    out = {}
    for f in FIELDS:
        val = fields.get(f)
        if val is None or str(val).strip() == "":
            out[f] = "NOT ANSWERABLE"
        else:
            out[f] = str(val).strip()
    return json.dumps(out, ensure_ascii=False)

def build_receipt_examples(items):
    examples = []
    for it in items:
        examples.append({
            "text": it["transcription"],
            "gold_json": normalize_fields(it["fields"])
        })
    return examples

receipts_gold = build_receipt_examples(items)

print("Num examples:", len(receipts_gold))
print("\nExample:")
print("TEXT:\n", receipts_gold[0]["text"][:400])
print("GOLD JSON:\n", receipts_gold[0]["gold_json"])


### 1.3 Train/test split

We split the labeled receipts into:
- training set (used to learn)
- test set (held out, used for evaluation)

We keep the split fixed using a random seed so results are reproducible.


In [ ]:
seed = 42
random.seed(seed)
np.random.seed(seed)

ds = Dataset.from_list(receipts_gold).train_test_split(test_size=0.2, seed=seed)
train_ds, test_ds = ds["train"], ds["test"]

print("Train size:", len(train_ds))
print("Test size:", len(test_ds))


## 2.Prepare model for fine-tunning

We choose an *instruct* causal language model (decoder-only), because our task is:
"given receipt text → produce JSON".

For a lab, we pick a model that is:
- small enough to train with LoRA
- still good at following instructions

To simplify things, we use a model that is not 'gated', meaning that we don't need to login or provide authentication keys to download. 

````bash
$ python scripts/download_models.py "unsloth/Llama-3.2-1B-Instruct" models/Llama-3.2-1B-Instruct
````


In [ ]:
# Small model (easier hardware, possibly of weaker quality):
base_model_path = "../../models/Llama-3.2-1B-Instruct"

### 2.1 Load tokenizer and model

We load:
- tokenizer: converts text ↔ tokens
- model: the base LLM we will adapt

For this, we use the same transformer classes as in our first transformers lab:
- AutoTokenizer: loads tokenizer
- AutoModelForCausalLM: loads llm model

We try loading the model in 4-bit to save memory (QLoRA-style loading).
If 4-bit loading fails in your environment, set `load_in_4bit=False`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(base_model_path, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

# configuration for quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

# loads model
model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    device_map="auto",
    quantization_config=bnb_config,
    dtype=torch.bfloat16,
)

model = prepare_model_for_kbit_training(model)

print("CUDA available:", torch.cuda.is_available())


### 2.2 Define the supervision format

Supervised fine-tuning learns by predicting the correct "assistant answer" given the prompt.

We enforce a strict contract:
- return ONLY JSON
- fixed keys: company, date, address, total
- if a field is missing: NOT ANSWERABLE

Consistency matters more than clever wording here.


In [ ]:
SYSTEM = "You extract structured information from receipts."

INSTRUCTION = """Extract the following fields from the receipt:
- company
- date
- address
- total

Return ONLY valid JSON with exactly these keys:
company, date, address, total

If a field is missing, use NOT ANSWERABLE for that field.
Do not add extra keys. Do not add explanations.
"""

def build_messages(receipt_text, system, instruction, target_json=None):
    msgs = [
        {"role": "system", "content": system},
        {"role": "user", "content": instruction + "\n\nRECEIPT:\n" + receipt_text},
    ]
    if target_json is not None:
        msgs.append({"role": "assistant", "content": target_json})
    return msgs


### 2.3 Turn each example into a single training string

We convert each (receipt_text, gold_json) into one chat-formatted text sequence:
system + user + assistant.

The trainer will learn to generate the assistant part.

In [ ]:
def format_for_sft(example):
    msgs = build_messages(example["text"],  SYSTEM, INSTRUCTION, example["gold_json"])
    # Use model-specific chat template if available:
    try:
        text = tokenizer.apply_chat_template(msgs, tokenize=False)
    except Exception:
        # Fallback: simple manual formatting
        text = (
            f"<s>[SYSTEM]\n{SYSTEM}\n"
            f"[USER]\n{INSTRUCTION}\n\nRECEIPT:\n{example['text']}\n"
            f"[ASSISTANT]\n{example['gold_json']}</s>"
        )
    return {"text_sft": text}

train_sft = train_ds.map(format_for_sft)
test_sft  = test_ds.map(format_for_sft)

# the text of the prompt is in the 'text_stf' field
print(train_sft[0]["text_sft"]) # truncated to the first 800 characters

## 3. Fine-tunning the model
With the model loaded, and data prepared, we can configure the fine-tunning process.

### 3.1 Add LoRA adapters (PEFT)

LoRA fine-tuning works by:
- freezing the base model weights
- adding small trainable matrices to selected layers (attention projections)

This means we train only a tiny fraction of parameters, making fine-tuning feasible in our lab.

#### Main LoRA parameters

| Parameter | Meaning | Value |
|-----------|---------|------:|
| `r` | Rank of the low-rank update matrices. Larger values increase capacity but also the number of trainable parameters. | `8` |
| `lora_alpha` | Scaling factor applied to the LoRA update. The effective update is roughly scaled by `α / r`. | `16` |
| `lora_dropout` | Dropout applied only to the LoRA branch during training to reduce overfitting. | `0.05` |
| `target_modules` | Layers where LoRA adapters are inserted. Here we modify only the **Query (Q)** and **Value (V)** projection matrices of the attention mechanism. | `["q_proj", "v_proj"]` |
| `bias` | Whether bias parameters are also fine-tuned. `"none"` keeps all biases frozen. | `"none"` |
| `task_type` | Specifies the downstream task so PEFT configures the adapters correctly. | `"CAUSAL_LM"` |

This trains only a very small fraction of the model while typically retaining most of the performance of full fine-tuning.

#### Target modules
For reference, the Llama model we are using, has these possible target modules: 
````
Transformer Block
├── Multi-Head Self-Attention
│   ├── q_proj
│   ├── k_proj
│   ├── v_proj
│   └── o_proj
└── Feed-Forward Network (MLP)
    ├── up_proj
    ├── gate_proj
    └── down_proj
````


In [ ]:
lora_cfg = LoraConfig(
    r=4,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    # We add the trainable matrices to Q and V (query and value)
    # We saw that in the paper this already showed good results
    target_modules=["q_proj", "v_proj"],
)

model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

### 3.2 Configure Trainer for fine-tuning (SFT)

We now configure a short fine-tuning session.
The model learns to produce the correct JSON, which should in principle should be *more reliably* than prompting alone.

In [ ]:
out_dir = "receipt_lora_adapter"

# training / fine tunning params
sft_cfg = SFTConfig(
    output_dir=out_dir,
    dataset_text_field="text_sft",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    num_train_epochs=1,
    logging_steps=10,
    save_steps=200,
    save_total_limit=2,
    report_to=[],
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_sft,
    args=sft_cfg,
)

### 3.3 Fine-tune model
We run the fine-tunning with LoRA based on the configuration above. In principle we can test different aspects: 
- effect of fine-tunning dataset
- effect of prompt
- effect of what parameters to fine-tune (Q, V, K, O)

In [ ]:
trainer.train()
trainer.save_model(out_dir)

print("Saved LoRA adapter to:", out_dir)

## 4. Evaluating the fine-tunned model

We can evaluate the model we trained in the "eval" jupyter notebook. We just load it by referencing the `out_dir`

#### LoRA number of parameters experiment

Train and evaluate two LoRA adapters:

1. LoRA applied to Query and Value projections.
2. LoRA applied to all attention projections: Query, Key, Value, and Output.

#### LoRA rank experiment

Train and evaluate two LoRA adapters with different ranks:

1. A low-rank configuration with `r=4`
2. A higher-rank configuration with `r=16`

Keep the ratio `lora_alpha / r` constant:

- for `r=4`, use `lora_alpha=8`
- for `r=16`, use `lora_alpha=32`

Use the same target modules in both configurations:


In [ ]:
# We can test the impact of the target modules (what to adapt)

lora_configs = [
    {
        "name": "q_v",
        "r": 8,
        "lora_alpha": 16,
        "target_modules": ["q_proj", "v_proj"],
    },
    {
        "name": "all_attention",
        "r": 8,
        "lora_alpha": 16,
        "target_modules": [
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
        ],
    },
]

# We can test two ranks (capacity)

lora_configs = [
    {
        "name": "rank_4",
        "r": 4,
        "lora_alpha": 8,
        "target_modules": ["q_proj", "v_proj"],
    },
    {
        "name": "rank_16",
        "r": 16,
        "lora_alpha": 32,
        "target_modules": ["q_proj", "v_proj"],
    },
]